# Notebook: BCI_19_CarDet_Prepara_Entrada
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_19_CarDet_Prepara_Entrada.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011897208
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/05/2022
* Descripcion: Prepara y carga informacion en tablas que son utilizadas como criterio de entrada, las cuales que en un inicio se cargaban con archivos productivos. Objetivo es dejar de depender de archivos productivos de otros sistemas.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/02/2025 
* Descripción: Prepara datos para nueva regla LIR      
***************************************************************************

**************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 09/04/2025 
* Descripción: Elimina logica de regla de negocio, salida de Lir.      
***************************************************************************

**************************************************************************
#### Mantención Nro: 3
* Autor: Gonzalo Arias (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 08/09/2025 
* Descripción: Se agrega lógica de 60 días de mora para determinar las operaciones renegociadas a deteriorar      
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_d00_segmentado
* {base_silver_x}.tbl_cd_d00_segmentado_pant
* {base_silver_x}.tbl_cd_curses
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_ope_condicion_ren 
* {base_silver_x}.tbl_cd_ope_condicion_mora
* {base_silver_x}.tbl_cd_ope_condicion_mora_pant
* {base_silver_x}.tbl_cd_ope_curse_bajo_mora
***************************************************************************


## Carga Dependencias

### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

# Notebook: Funciones_Comunes
**************************************************************************

## Informacion del Notebook 

### Encabezado
**************************************************************************
* Nombre: Funciones_Comunes.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/3570959530595695
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 23/09/2023
* Descripcion: Notebook con funciones genéricas que pueden ser usadas por otros notebooks.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 15/02/2025 
* Descripción: Se cambio el metodo de cancelacion utilizando el comando (raise) y se incorporada la funcion de ir a buscar el ultimo dia calendario. Tambien se agrego una nueva funcion (obtener_estados_tablas).  
***************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 25/04/2025 
* Descripción: Se modifico la funcion extension_archivos para que cuando la vigencia sea previa, asigne extencion .PRV.  
***************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 08/07/2025 
* Descripción: Se realiza una mejora en la funcion mostrar_variacion_criterio.  
***************************************************************************

## Carga librerias

...

### obtener archivo periodo anterior

### concatena archivos

###primer_dia_mes_sig

###Calcula fecha X meses atras

## FIN definición de funciones

## Parámetros

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","02-Nombre BD Silver:")

fecha_x = dbutils.widgets.get("fecha_w") 
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.Fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


Fecha de Proceso actual: [fecha_x] 20250930
Nombre BD Silver: [base_silver_x] dsr_gld_bciwork_db


### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
#parametria interna notebook
p_tip_prod='ACT'
p_tip_cart='CON','COM'
p_cod_ren='S','C'

print(f"p_tip_prod: {p_tip_prod}")
print(f"p_tip_cart: {p_tip_cart}")
print(f"p_cod_ren: {p_cod_ren}")


p_tip_prod: ACT
p_tip_cart: ('CON', 'COM')
p_cod_ren: ('S', 'C')


###Genera tabla pivote con la informacion necesaria para el calculo de deterioro
---------------------------
* La tabla pivote es el d00segmentado del periodo actual

####Extrae datos de tablas nativas

In [0]:
paso_query100 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cd_ope_d00_pact as
SELECT 
/*DATOS DE: tbl_cd_d00_segmentado*/
 A.periodo_cierre
,A.fecha_cierre
,A.tipo_proceso
,A.macro_sistema
,A.tipo_cartera
,A.tipo_producto
,A.segmento
,A.cod_proceso
,A.sistema
,A.tipo_credito
,A.operacion
,A.tipo_operacion
,A.cuenta_contable
,A.saldo_cartera_venc
,A.saldo_total_ifrs
,A.cod_renegociado
,A.fecha_otorgamiento
,A.fecha_inicio_mora
,A.rut_cliente
,A.dv_rut_cliente
,A.cod_calificacion
,A.dias_mora
,A.fecha_cart_venc
,CASE
    WHEN A.fecha_otorgamiento = 19000101 THEN 0
    WHEN A.fecha_otorgamiento = 0 THEN 0
    WHEN to_date(cast(A.fecha_otorgamiento AS string), 'yyyyMMdd') <= (to_date(SUBSTRING(cast(A.fecha_cierre AS string),1,6)||'01', 'yyyyMMdd')) THEN 0
    ELSE datediff(to_date(cast(A.fecha_otorgamiento AS string), 'yyyyMMdd'),  (to_date(SUBSTRING(cast(A.fecha_cierre AS string),1,6)||'01', 'yyyyMMdd'))) + 1
 END AS ope_dias_curse_pact

/*DATOS DE: tbl_cd_d00_segmentado_pant*/
,CASE WHEN B.operacion is not null THEN 1 ELSE 0 END AS flag_existe_operacion_pant
,IFNULL(B.ind_cartdet,'N')  AS ope_ind_cartdet_pant
,IFNULL(B.fecha_cart_venc,0) AS ope_fecha_cart_venc_pant

/*DATOS DE: tbl_cd_curses*/
,CASE WHEN C.operacion is not null THEN 1 ELSE 0 END AS flag_existe_operacion_rfz
,IFNULL(C.fec_proc,0) AS ope_fec_proc_rfz

/*DATOS DE: tbl_cd_segmentacion_cliente*/
,CASE WHEN D.rut_cliente is not null THEN 1 ELSE 0 END AS flag_existe_cliente_seg_cli
,IFNULL(D.segmento_cliente,'SD') AS cli_segmento_cliente

/*DATOS DE: tbl_cd_cliente_consolidado*/
,CASE WHEN E.rut_cliente is not null THEN 1 ELSE 0 END AS flag_existe_cliente_cli_con
,IFNULL(E.calificacion_bci,'SD') AS cli_calificacion_bci

/*DATOS DE: tbl_cd_cliente_lir*/
,CASE WHEN F.rut_cliente is not null THEN 1 ELSE 0 END AS flag_existe_cliente_lir
,IFNULL(F.fecha_informada,0) AS cli_fecha_informada_lir

/*DATOS DE: tbl_cd_cliente_det_ssff*/
,CASE WHEN G.rut_cliente is not null THEN 1 ELSE 0 END AS flag_existe_cliente_ssff

/*DATOS DE: tbl_cd_cliente_det_fact*/
,CASE WHEN H.rut_cliente is not null THEN 1 ELSE 0 END AS flag_existe_cliente_fact

FROM
  {base_silver_x}.tbl_cd_d00_segmentado A
LEFT JOIN
  {base_silver_x}.tbl_cd_d00_segmentado_pant B
ON   A.operacion = B.operacion  AND A.sistema = B.sistema  
LEFT JOIN
   {base_silver_x}.tbl_cd_curses C
ON   trim(A.operacion) = trim(C.operacion) AND trim(A.sistema) = trim(C.cod_sistema)
LEFT JOIN
    {base_silver_x}.tbl_cd_segmentacion_cliente D
ON A.rut_cliente = D.rut_cliente 
LEFT JOIN
    {base_silver_x}.tbl_cd_cliente_consolidado E
ON A.rut_cliente = E.rut_cliente
LEFT JOIN
    {base_silver_x}.tbl_cd_cliente_lir F
ON A.rut_cliente = F.rut_cliente
LEFT JOIN
    {base_silver_x}.tbl_cd_cliente_det_ssff G
ON A.rut_cliente = G.rut_cliente
LEFT JOIN
    {base_silver_x}.tbl_cd_cliente_det_fact H
ON A.rut_cliente = H.rut_cliente
"""


In [0]:
sql_safe(paso_query100)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cd_ope_d00_pact as
SELECT 
/*DATOS DE: tbl_cd_d00_segmentado*/
 A.periodo_cierre
,A.fecha_cierre
,A.tipo_proceso
,A.macro_sistema
,A.tipo_cartera
,A.tipo_producto
,A.segmento
,A.cod_proceso
,A.sistema
,A.tipo_credito
,A.operacion
,A.tipo_operacion
,A.cuenta_contable
,A.saldo_cartera_venc
,A.saldo_total_ifrs
,A.cod_renegociado
,A.fecha_otorgamiento
,A.fecha_inicio_mora
,A.rut_cliente
,A.dv_rut_cliente
,A.cod_calificacion
,A.dias_mora
,A.fecha_cart_venc
,CASE
    WHEN A.fecha_otorgamiento = 19000101 THEN 0
    WHEN A.fecha_otorgamiento = 0 THEN 0
    WHEN to_date(cast(A.fecha_otorgamiento AS string), 'yyyyMMdd') <= (to_date(SUBSTRING(cast(A.fecha_cierre AS string),1,6)||'01', 'yyyyMMdd')) THEN 0
    ELSE datediff(to_date(cast(A.fecha_otorgamiento AS string), 'yyyyMMdd'),  (to_date(SUBSTRING(cast(A.fecha_cierre AS string),1,6)||'01', 'yyyyMMdd'))) + 1
 END AS ope_dias_curse_pact

/*DATOS DE: tbl_cd_d00_segmentado_pant*/
,CAS

DataFrame[]

#### Cae Excepciones Imcumplimiento: solo operaciones

In [0]:
paso_query110 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_ope_cae_incl AS
SELECT 
  trim(b.operacion)                   AS operacion,
  trim(b.tipo_operacion)              AS tipo_operacion,
  trim(b.anio_lic)                    AS anio_lic,
  count(b.operacion)                  AS cant
FROM
   {base_silver_x}.tbl_cd_cae_ope_det_incumplimiento b
group by 1,2,3
"""  

In [0]:
sql_safe(paso_query110)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_ope_cae_incl AS
SELECT 
  trim(b.operacion)                   AS operacion,
  trim(b.tipo_operacion)              AS tipo_operacion,
  trim(b.anio_lic)                    AS anio_lic,
  count(b.operacion)                  AS cant
FROM
   dsr_gld_bciwork_db.tbl_cd_cae_ope_det_incumplimiento b
group by 1,2,3



DataFrame[]

####Calcula maximo dias de mora para renegociados
---------------------------
* Calculo exclusivo para el deterioro de operaciones renegociadas

#####Calcula maximo dias de curse del cliente en periodo actual 
---------------------------
* los dias de curse del periodo actual, para operaciones renegociadas, se consideran como mora, para el calculo de deterioro renegociado
* estos dias de curse (mora) se deben sumar a los dias de mora del periodo anterior para obtener los clientes renegociados con 60 o mas dias de mora.

In [0]:
paso_query131 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_cli_mor_ren_pact as
select 
  a.fecha_cierre,
  a.rut_cliente,
  max(a.ope_dias_curse_pact) as max_dias_curse_pact
from 
   tmp_EXT_tbl_cd_ope_d00_pact  a
where
   flag_existe_operacion_pant=0 
  AND TRIM(A.tipo_producto) = '{p_tip_prod}' 
  AND TRIM(A.tipo_cartera) IN {p_tip_cart} 
  AND TRIM(A.cod_renegociado) IN {p_cod_ren}
group by 1,2
"""


In [0]:
sql_safe(paso_query131)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_cli_mor_ren_pact as
select 
  a.fecha_cierre,
  a.rut_cliente,
  max(a.ope_dias_curse_pact) as max_dias_curse_pact
from 
   tmp_EXT_tbl_cd_ope_d00_pact  a
where
   flag_existe_operacion_pant=0 
  AND TRIM(A.tipo_producto) = 'ACT' 
  AND TRIM(A.tipo_cartera) IN ('CON', 'COM') 
  AND TRIM(A.cod_renegociado) IN ('S', 'C')
group by 1,2



DataFrame[]

##### Calculo maximo dias de mora del cliente en periodo anterior
---------------------------
* Estos dias de mora, en conjunto con dias de curse (mora) de las operaciones renegociadas del periodo actual, se suman para el calculo final



In [0]:
paso_query132 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_cli_mor_ren_pant as
select 
  a.fecha_cierre,
  a.rut_cliente,
  max(a.dias_mora) as max_dia_mora
from 
   {base_silver_x}.tbl_cd_d00_segmentado_pant  a
group by 1,2
"""


In [0]:
sql_safe(paso_query132)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_cli_mor_ren_pant as
select 
  a.fecha_cierre,
  a.rut_cliente,
  max(a.dias_mora) as max_dia_mora
from 
   dsr_gld_bciwork_db.tbl_cd_d00_segmentado_pant  a
group by 1,2



DataFrame[]

##### Calculo maximo dias mora cliente periodo actual 
---------------------------
* Calculo maximo dias de mora en periodo actual considerando los dias de curse de las operaciones renegociadas nuevas del periodo actual mas
* el maximo dias de mora del periodo anterior de todas las operaciones del cliente.



In [0]:
paso_query133 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_cli_mor_ren as
select
  a.fecha_cierre, 
  a.rut_cliente, 
  a.max_dias_curse_pact                              as max_dias_curse_pact,
  b.max_dia_mora                                     as max_dia_mor_pant, 
  (a.max_dias_curse_pact + ifnull(b.max_dia_mora,0)) as max_dia_mora
from
 tmp_RES_tbl_cd_cli_mor_ren_pact a
left join
 tmp_RES_tbl_cd_cli_mor_ren_pant b
on a.rut_cliente = b.rut_cliente
"""

In [0]:
sql_safe(paso_query133)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cd_cli_mor_ren as
select
  a.fecha_cierre, 
  a.rut_cliente, 
  a.max_dias_curse_pact                              as max_dias_curse_pact,
  b.max_dia_mora                                     as max_dia_mor_pant, 
  (a.max_dias_curse_pact + ifnull(b.max_dia_mora,0)) as max_dia_mora
from
 tmp_RES_tbl_cd_cli_mor_ren_pact a
left join
 tmp_RES_tbl_cd_cli_mor_ren_pant b
on a.rut_cliente = b.rut_cliente



DataFrame[]

##Genera tablon de salida con todos los datos necesarios para evaluar la entrada a cartera deteriorada

In [0]:
paso_query200 = f"""
CREATE OR REPLACE TEMPORARY VIEW tbl_cd_ope_condicion_deterioro as
SELECT
   A.periodo_cierre
  ,A.fecha_cierre
  ,A.tipo_proceso
  ,A.macro_sistema
  ,A.tipo_cartera
  ,A.tipo_producto
  ,A.segmento
  ,A.cod_proceso
  ,A.sistema
  ,A.tipo_credito
  ,A.operacion
  ,A.tipo_operacion
  ,A.cuenta_contable
  ,A.saldo_cartera_venc
  ,A.saldo_total_ifrs
  ,A.cod_renegociado
  ,A.fecha_otorgamiento
  ,A.fecha_inicio_mora
  ,A.rut_cliente
  ,A.dv_rut_cliente
  ,A.cod_calificacion
  ,A.dias_mora
  ,A.fecha_cart_venc
  ,A.ope_dias_curse_pact
  ,A.flag_existe_operacion_pant
  ,A.ope_ind_cartdet_pant
  ,A.ope_fecha_cart_venc_pant
  ,A.flag_existe_operacion_rfz
  ,A.ope_fec_proc_rfz
  ,A.flag_existe_cliente_seg_cli
  ,A.cli_segmento_cliente
  ,A.flag_existe_cliente_cli_con
  ,A.cli_calificacion_bci
  ,A.flag_existe_cliente_lir
  ,A.cli_fecha_informada_lir
  ,A.flag_existe_cliente_ssff
  ,A.flag_existe_cliente_fact
  ,IFNULL(B.max_dias_curse_pact,0) AS max_dias_curse_pact
  ,IFNULL(B.max_dia_mor_pant,0) AS max_dia_mor_pant
  ,IFNULL(B.max_dia_mora,0) AS max_dia_mora_ren
  ,CASE WHEN C.operacion IS NOT NULL AND substring(cast(A.fecha_otorgamiento as string),1,4) = trim(C.anio_lic)   THEN 1 ELSE 0 END AS flag_cae_excepcion

FROM
  tmp_EXT_tbl_cd_ope_d00_pact A
LEFT JOIN
  tmp_RES_tbl_cd_cli_mor_ren B
ON A.rut_cliente=B.rut_cliente
LEFT JOIN
  tmp_RES_tbl_cd_ope_cae_incl C
ON
    trim(A.operacion)  = trim(C.operacion) AND TRIM(A.tipo_operacion) = TRIM(C.tipo_operacion)    
"""


In [0]:
sql_safe(paso_query200)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tbl_cd_ope_condicion_deterioro as
SELECT
   A.periodo_cierre
  ,A.fecha_cierre
  ,A.tipo_proceso
  ,A.macro_sistema
  ,A.tipo_cartera
  ,A.tipo_producto
  ,A.segmento
  ,A.cod_proceso
  ,A.sistema
  ,A.tipo_credito
  ,A.operacion
  ,A.tipo_operacion
  ,A.cuenta_contable
  ,A.saldo_cartera_venc
  ,A.saldo_total_ifrs
  ,A.cod_renegociado
  ,A.fecha_otorgamiento
  ,A.fecha_inicio_mora
  ,A.rut_cliente
  ,A.dv_rut_cliente
  ,A.cod_calificacion
  ,A.dias_mora
  ,A.fecha_cart_venc
  ,A.ope_dias_curse_pact
  ,A.flag_existe_operacion_pant
  ,A.ope_ind_cartdet_pant
  ,A.ope_fecha_cart_venc_pant
  ,A.flag_existe_operacion_rfz
  ,A.ope_fec_proc_rfz
  ,A.flag_existe_cliente_seg_cli
  ,A.cli_segmento_cliente
  ,A.flag_existe_cliente_cli_con
  ,A.cli_calificacion_bci
  ,A.flag_existe_cliente_lir
  ,A.cli_fecha_informada_lir
  ,A.flag_existe_cliente_ssff
  ,A.flag_existe_cliente_fact
  ,IFNULL(B.max_dias_curse_pact,0) AS max_dias_curse_pact
  ,IFNU

DataFrame[]

## Carga Tablas de Salidas 
--------------------------------------
* carga resultados a tablas de salidas del notebook

#### Reproceso (Elimina registros en caso de reprocesos)

#####TRUNCATE tbl_cd_ope_condicion_deterioro

In [0]:
paso_query300 = f""" TRUNCATE TABLE {base_silver_x}.tbl_cd_ope_condicion_deterioro """

In [0]:
sql_safe(paso_query300)

sql_safe: query ->  TRUNCATE TABLE dsr_gld_bciwork_db.tbl_cd_ope_condicion_deterioro 


DataFrame[]

#### Inserta registros tabla de salida

#####INSERT INTO tbl_cd_ope_condicion_deterioro

In [0]:
paso_query400 = f"""
INSERT INTO {base_silver_x}.tbl_cd_ope_condicion_deterioro
SELECT 
   A.periodo_cierre
  ,A.fecha_cierre
  ,A.tipo_proceso
  ,A.macro_sistema
  ,A.tipo_cartera
  ,A.tipo_producto
  ,A.segmento
  ,A.cod_proceso
  ,A.sistema
  ,A.tipo_credito
  ,A.operacion
  ,A.tipo_operacion
  ,A.cuenta_contable
  ,A.saldo_cartera_venc
  ,A.saldo_total_ifrs
  ,A.cod_renegociado
  ,A.fecha_otorgamiento
  ,A.fecha_inicio_mora
  ,A.rut_cliente
  ,A.dv_rut_cliente
  ,A.cod_calificacion
  ,A.dias_mora
  ,A.fecha_cart_venc
  ,A.ope_dias_curse_pact
  ,A.flag_existe_operacion_pant
  ,A.ope_ind_cartdet_pant
  ,A.ope_fecha_cart_venc_pant
  ,A.flag_existe_operacion_rfz
  ,A.ope_fec_proc_rfz
  ,A.flag_existe_cliente_seg_cli
  ,A.cli_segmento_cliente
  ,A.flag_existe_cliente_cli_con
  ,A.cli_calificacion_bci
  ,A.flag_existe_cliente_lir
  ,A.cli_fecha_informada_lir
  ,A.flag_existe_cliente_ssff
  ,A.flag_existe_cliente_fact
  ,A.max_dias_curse_pact
  ,A.max_dia_mor_pant
  ,A.max_dia_mora_ren
  ,A.flag_cae_excepcion
FROM
  tbl_cd_ope_condicion_deterioro  A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.fecha_cierre DESC) =1  
""" 

In [0]:
sql_safe(paso_query400)

sql_safe: query -> 
INSERT INTO dsr_gld_bciwork_db.tbl_cd_ope_condicion_deterioro
SELECT 
   A.periodo_cierre
  ,A.fecha_cierre
  ,A.tipo_proceso
  ,A.macro_sistema
  ,A.tipo_cartera
  ,A.tipo_producto
  ,A.segmento
  ,A.cod_proceso
  ,A.sistema
  ,A.tipo_credito
  ,A.operacion
  ,A.tipo_operacion
  ,A.cuenta_contable
  ,A.saldo_cartera_venc
  ,A.saldo_total_ifrs
  ,A.cod_renegociado
  ,A.fecha_otorgamiento
  ,A.fecha_inicio_mora
  ,A.rut_cliente
  ,A.dv_rut_cliente
  ,A.cod_calificacion
  ,A.dias_mora
  ,A.fecha_cart_venc
  ,A.ope_dias_curse_pact
  ,A.flag_existe_operacion_pant
  ,A.ope_ind_cartdet_pant
  ,A.ope_fecha_cart_venc_pant
  ,A.flag_existe_operacion_rfz
  ,A.ope_fec_proc_rfz
  ,A.flag_existe_cliente_seg_cli
  ,A.cli_segmento_cliente
  ,A.flag_existe_cliente_cli_con
  ,A.cli_calificacion_bci
  ,A.flag_existe_cliente_lir
  ,A.cli_fecha_informada_lir
  ,A.flag_existe_cliente_ssff
  ,A.flag_existe_cliente_fact
  ,A.max_dias_curse_pact
  ,A.max_dia_mor_pant
  ,A.max_dia_mora_ren


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")